## The corpus

The corpus holds several Palestinian Targum manuscripts:
Targum Pseudo-Jonathan, the Fragment Targums, Cairo
Genizah fragments, Targum Neofiti as a 'base' text (only unmarked text segments, no variants, no interlinear or in-line additions, retain the raw text) and as a 'full' text (all segments side-by-side). The text is Jewish Palestinian Aramaic, carried in ETCBC transliteration and
in Aramaic square script.

The object hierarchy is `text > book > chapter > verse > segment > word_group > word`, and
`word` is the slot type. `word_group` is included for now as a feature meant to ensure consistency across the hierarchy as certain data modeling issues are resolved. Segment `type`
features record that state, with values including `unmarked_text`, `variant`, `addition`,
`editorial_deletion`, `broken_text` and `interlinear`. Segments and words both carry `start_idx` and `end_idx`,
one-based character offsets into the `line` feature of the verse that contains them; a segment's
are the minimum and the maximum of the offsets of the words it holds, so the two tiers cannot
disagree.

Two features record what the publication boundary does to a word. `source` is `word` where
morphology was placed on it, `word_group` where the group carries a surface no morphology points
at, `none` where the manuscript has a word CAL never analysed, and `segment` where the word stands
for a segment that no word of the manuscript reaches, carrying that segment's own text so the
segment can be published at all. `part` counts the pieces of a word that an editorial boundary
cuts in two: the pieces share their `word_group` and its `id`, each sits in the segment its own
characters fall in, and a word with `part` 1 and no sibling is whole. Because a word group's
pieces are two nodes, the tests below that speak about a word group aggregate over the nodes
sharing one `id` rather than over nodes.

## Baseline tests

Tests 1 to 5 go through the object hierarchy upward, and ask
whether a tier of nodes reconstructs the tier above it, for instance do all the segments of a given text construct the verse line itself. Test 4 also takes the segment offsets and resolves them
against the line they index. Test 6 assesses if the stored ordinals agree with the order the
dataset publishes.

Each test reports per corpus, because the five corpora differ substantially in how much
morphology the source material provides.

### Reading the results

Tests 1, 3, 4 and 5 report substantive figures that vary by corpus, and their variation is a
property of the source material. The failures are concentrated in two identifiable phenomena that the
second table of test 1 isolates: surface forms whose morphology accounts for part of the word,
and a diacritic written one way at the word tier and another way at the word group tier.

Test 2 reports a figure that describes the publication boundary rather than the corpus. A word
straddling two editorial states is contained by one segment, which leaves the other segment without
a group that spells it, and the same effect limits the proportion of segment spans whose slice
returns the segment and the proportion of verses whose segments spell the line, both of them
reported at test 4.

In [1]:
# Loads the dataset and reports its provenance. The version string, the version of the XML the
# dataset was compiled from and the compilation timestamp are read from the metadata of the
# `otype` feature, which every Text-Fabric dataset carries. The node counts printed here are the
# quantities every later test is measured against.
#
# The dataset is this repository's own `tf/`, reached through the org and repo `.env` names and
# `programs/xml_to_tf.py` writes under. It used to be `etcbc/targum:clone`, which is a separate
# hand-maintained directory: on 2026-09-28 that copy was three days behind this repository and
# every figure in this notebook was a measurement of it rather than of the build the pipeline had
# just produced. Reading the names from the environment rather than spelling them out is what
# stops the two drifting again.

import os
from pathlib import Path

from dotenv import load_dotenv
from tf.app import use

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config/xml.yml").exists())
load_dotenv(REPO_ROOT / ".env")
ORG, REPO = os.environ["TF_ORG"], os.environ["TF_REPO"]

A = use(f"{ORG}/{REPO}:clone", checkout="clone", version="0.0.1-alpha", hoist=globals())

meta = A.api.TF.features["otype"].metaData


**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
text,68,7282.59,100
book,86,5758.33,100
chapter,1038,477.09,100
verse,19595,25.27,100
segment,55197,8.97,100
word_group,370404,1.34,100
word,495216,1.00,100


In [2]:
# Provenance. Three assertions, and the dataset is stale unless all three hold.
#
# Test 7 of `sql_tf_comparison.ipynb` compares the dataset to the XML it was compiled from, and it
# would have passed throughout the episode these checks exist because of: the XML was internally
# consistent with the dataset and both were older than the database. So none of these three asks
# whether the dataset matches its XML.
#
#   1  xmlVersion against the newest directory under `xml/`. A dataset compiled from an older XML
#      than the pipeline last wrote is behind the database by at least that much.
#   2  the feature descriptions against `config/xml.yml`. This is the check that caught the
#      2026-09-25 build: `start_idx` had been redeclared for two tiers in the config and the
#      dataset still carried the one-tier wording, which dated it more precisely than any
#      timestamp did.
#   3  the node counts against the five publication views, if the database is reachable. `word` is
#      the slot type, so its count is the row count of the five `*_tf` views together.
#
# A failure here is not a reason to read on. Every number below is a measurement of whatever this
# cell loaded.

import re
import yaml

FRESH = True


def stale(what, got, want):
    global FRESH
    FRESH = False
    print(f"  STALE  {what}\n           dataset: {got}\n           source:  {want}")


xml_versions = sorted(p.name for p in (REPO_ROOT / "xml").iterdir() if p.is_dir())
newest_xml = xml_versions[-1] if xml_versions else None
print(f"xmlVersion {meta.get('xmlVersion')}  written {meta.get('dateWritten')}")
print(f"newest xml/ {newest_xml}  (of {len(xml_versions)})")
if newest_xml and meta.get("xmlVersion") != newest_xml:
    stale("the dataset was compiled from an older XML than xml/ holds",
          meta.get("xmlVersion"), newest_xml)

declared = yaml.safe_load((REPO_ROOT / "config/xml.yml").read_text())["featureDescriptions"]


def norm(s):
    return re.sub(r"\s+", " ", (s or "")).strip()


for name, want in declared.items():
    feat = A.api.TF.features.get(name)
    if feat is None:
        continue
    got = feat.metaData.get("description")
    if norm(got) != norm(want):
        stale(f"feature {name!r} carries a description config/xml.yml no longer declares",
              norm(got)[:110], norm(want)[:110])

counts = {t: n for t, n in ((t, len(list(F.otype.s(t)))) for t in F.otype.all)}
print("nodes:", ", ".join(f"{t} {counts[t]:,}" for t in F.otype.all))

try:
    import psycopg

    with psycopg.connect(
        dbname=os.environ["DB_NAME"], user=os.environ["DB_USER"],
        password=os.environ["DB_PASSWORD"], host=os.environ["DB_HOST"],
        port=os.environ["DB_PORT"],
    ) as _c, _c.cursor() as _cur:
        total = 0
        for v in ("cairo_genizah_tf", "frag_targ_tf", "neofiti_base_raw_tf",
                  "neofiti_full_tf", "psj_tf"):
            _cur.execute(f"SELECT count(*) FROM {v}")
            total += _cur.fetchone()[0]
    print(f"slots in the dataset {counts['word']:,}  rows in the five *_tf views {total:,}")
    if counts["word"] != total:
        stale("the slot count is not the row count of the publication views",
              f"{counts['word']:,} word nodes", f"{total:,} view rows")
except Exception as exc:  # the database is optional here; the first two checks are not
    print(f"(node counts not checked against the views: {type(exc).__name__}: {exc})")

print()
print("dataset is current" if FRESH else "DATASET IS STALE — rebuild before reading on")
assert FRESH, "stale dataset: see the report above"


xmlVersion 2026-09-29  written 2026-09-30T01:09:04+00:00Z
newest xml/ 2026-09-29  (of 1)
nodes: text 68, book 86, chapter 1,038, verse 19,595, segment 55,197, word_group 370,404, word 495,216


slots in the dataset 495,216  rows in the five *_tf views 495,216

dataset is current


In [3]:
# Report the feature inventory and the distribution of the three features that the tests depend
# on for their interpretation. `source` determines which words the alignment is answerable for,
# `reading` identifies the readings that compete for one position, and the segment `type` records
# the editorial state of a run of the line.
#

FEATURES = sorted(f for f in A.api.TF.features if not f.startswith("o") and f != "otext")
for otype in F.otype.all:
    sample = F.otype.s(otype)[:400]
    present = sorted(f for f in FEATURES
                     if hasattr(A.api.F, f) and any(A.api.Fs(f).v(n) is not None for n in sample))
    print(f"{otype:<11} {', '.join(present)}")
print()
print("source     ", dict(F.source.freqList()))
print("reading    ", dict(F.reading.freqList()))
print("segment    ", dict(F.type.freqList()))
print()
for fmt in sorted(T.formats):
    print(f"  {fmt:<16} {T.text(range(1, 9), fmt=fmt)}")


text        text, view
book        book, id
chapter     chapter
verse       line, line_utf8, uid, verse
segment     end_idx, id, rank, segment, segment_utf8, start_idx, type
word_group  g_cons, g_cons_utf8, g_voc, g_voc_utf8, id, rank, reading
word        cons, cons_utf8, definition, description, end_idx, gloss, gn, id, lex, nu, part, root, source, sp, st, start_idx, text, trailer, uid, voc, voc_utf8, vs

source      {'word': 487996, 'word_group': 5914, 'none': 1304, 'segment': 2}


reading     {0: 362386, 1: 7763, 2: 253, 3: 2}
segment     {'unmarked_text': 33953, 'variant': 11051, 'addition': 3414, 'marked_text': 3094, 'editorial_deletion': 2266, 'broken_text': 841, 'interlinear': 237, 'scribal_deletion': 213, 'incomplete': 122, 'source': 4}

  text-cons        איליין פיתגמיא די מליל משׁה עם כל ישׁראל 
  text-orig-full   איליין פיתגמיא די מליל משׁה עם כל ישׁראל 
  text-trans-cons  >JLJJn PJTGMJ> DJ MLJL MCH <m KL JCR>L 
  text-trans-full  >JLJJn PJTGMJ> DJ MLJL MCH <m KL JCR>L 


In [4]:
# Defines the material shared by every test below.
#
# `letters` is the comparison class. The tests compare strings that were produced by different
# parts of the upstream pipeline.
#
# `verse_nodes` keys each verse by the pair (manuscript, uid). The `uid` feature is unique within
# one corpus and repeats across corpora, so the manuscript name is needed to make the key unique.
# Two verses of the Genizah material occupy more than one verse node, because the order in which
# the dataset publishes their words does not keep the rows of those verses contiguous, and the
# tests therefore aggregate over the list of nodes that share a key.
#
# `table` prints one row per corpus with a total, and takes derived percentage columns as
# (label, numerator, denominator) triples.

import re
from collections import Counter, defaultdict

_WS = re.compile(r"\s+")


def letters(s):
    return _WS.sub("", s or "")


CORPORA = ["cairo_genizah", "frag_targ", "neofiti_base_raw", "neofiti_full", "psj"]
_BY_NAME = {"neofiti_base": "neofiti_base_raw", "neofiti_full": "neofiti_full",
            "pseudo_jonathan": "psj"}


def corpus_of(text_node):
    name = F.text.v(text_node)
    if name.startswith("cairo"):
        return "cairo_genizah"
    if name.startswith("fragment_targum"):
        return "frag_targ"
    return _BY_NAME[name]


verse_nodes = defaultdict(list)
corpus_of_verse = {}
for _t in F.otype.s("text"):
    _name, _corpus = F.text.v(_t), corpus_of(_t)
    for _v in L.d(_t, otype="verse"):
        verse_nodes[(_name, F.uid.v(_v))].append(_v)
        corpus_of_verse[(_name, F.uid.v(_v))] = _corpus


def table(title, rows, cols, pct=(), key="corpus"):
    keys = [k for k in CORPORA if k in rows] + [k for k in rows if k not in CORPORA]
    totals = Counter()
    for k in keys:
        totals.update(rows[k])
    head = [key, *cols, *(lbl for lbl, _, _ in pct)]
    body = []
    for k in [*keys, "TOTAL"]:
        r = totals if k == "TOTAL" else rows[k]
        body.append([k, *(f"{r[c]:,}" for c in cols),
                     *(f"{100.0 * r[n] / r[d]:.2f}" if r[d] else "-" for _, n, d in pct)])
    w = [max(len(str(x)) for x in col) for col in zip(head, *body)]
    rule = "  ".join("-" * i for i in w)
    print(title)
    print("  " + "  ".join(h.rjust(i) for h, i in zip(head, w)))
    print("  " + rule)
    for line in body[:-1]:
        print("  " + "  ".join(str(x).rjust(i) for x, i in zip(line, w)))
    print("  " + rule)
    print("  " + "  ".join(str(x).rjust(i) for x, i in zip(body[-1], w)))


results = {}
print(f"{len(verse_nodes):,} verses distributed over "
      f"{sum(len(v) for v in verse_nodes.values()):,} verse nodes")

19,587 verses distributed over 19,595 verse nodes


In [5]:
# TEST 1. Composition of a word group from its words.
#
# Method: for every word group, concatenate the `voc` feature of the `word` nodes it contains,
# in slot order, and compare the result to the group's own `g_voc` feature under the letters
# comparison.
#
# A word group is not a node here, it is an `id`. Where an editorial boundary cuts a word in
# two, each piece sits in a different segment and the serialiser therefore opens a second
# `word_group` node for it — both carrying the whole CAL word as `g_voc`, each holding only its
# own words. Read node by node, every such group fails twice for no defect. Read by `id`, which
# is what this does, the statement is the one intended: the morphology of a CAL word accounts
# for that word.
#
# What it measures: whether the morphological decomposition accounts for the surface form it was
# derived from. A group whose words spell it exactly is one where every character of the
# manuscript word is claimed by some morpheme.
#
# Interpretation: `analyzed` restricts the population to the groups all of whose words carry
# morphology, which is the set the decomposition is answerable for. The second table classifies
# the failures. `reordered` would indicate the same letters in a different order, which is a
# alignment fault. `words_shorter` indicates that the morphology covers part of the
# surface form and is a property of the source material. `dot_vs_mark` indicates that the two
# strings differ only in whether a Genizah diacritic is written as a full stop or as the
# combining mark U+0740.

test1 = defaultdict(Counter)
group_failures = []
for _k, _vns in verse_nodes.items():
    _c = corpus_of_verse[_k]
    # The pieces of one word group, in node order, keyed by the group's own id.
    _by_id = defaultdict(list)
    for _v in _vns:
        for _g in L.d(_v, otype="word_group"):
            _by_id[F.id.v(_g)].append(_g)
    for _gid, _nodes in _by_id.items():
        ws = [w for _g in _nodes for w in L.d(_g, otype="word")]
        built = "".join(F.voc.v(w) or "" for w in ws)
        target = F.g_voc.v(_nodes[0]) or ""
        analysed = {F.source.v(w) for w in ws} == {"word"}
        ok = letters(built) == letters(target)
        _r = test1[_c]
        _r["groups"] += 1
        _r["words"] += len(ws)
        _r["split_groups"] += len(_nodes) > 1
        _r["words_spell_group"] += ok
        _r["differ"] += not ok
        _r["analysed"] += analysed
        _r["analysed_spelled"] += analysed and ok
        if not ok:
            group_failures.append((_c, _nodes[0], target, built))
            _b, _t = letters(built), letters(target)
            _r["reordered"] += sorted(_b) == sorted(_t)
            _r["words_shorter"] += len(_b) < len(_t)
            _r["words_longer"] += len(_b) > len(_t)
            _r["dot_vs_mark"] += _b.replace("݀", ".") == _t

table("TEST 1  the words of a word group spell the group", test1,
      ["groups", "split_groups", "words", "words_spell_group", "differ", "analysed",
       "analysed_spelled"],
      pct=[("%", "words_spell_group", "groups"), ("% of analysed", "analysed_spelled", "analysed")])
print()
table("TEST 1  the composition of the failures", test1,
      ["differ", "reordered", "words_shorter", "words_longer", "dot_vs_mark"])
results["1 words compose the word group"] = (sum(test1[c]["groups"] for c in CORPORA),
                                             sum(test1[c]["words_spell_group"] for c in CORPORA))


TEST 1  the words of a word group spell the group
            corpus   groups  split_groups    words  words_spell_group  differ  analysed  analysed_spelled      %  % of analysed
  ----------------  -------  ------------  -------  -----------------  ------  --------  ----------------  -----  -------------
     cairo_genizah   12,451         1,430   18,085             12,428      23    12,278            12,257  99.82          99.83
         frag_targ   32,012           650   43,584             31,995      17    31,976            31,972  99.95          99.99
  neofiti_base_raw   96,108         1,403  132,969             95,653     455    95,191            95,180  99.53          99.99
      neofiti_full  115,679         1,446  157,316            115,477     202   110,550           110,468  99.83          99.93
               psj  104,763           383  143,262            104,737      26   104,735           104,731  99.98         100.00
  ----------------  -------  ------------  -------  --

In [6]:
# Displays four of the word groups that test 1 reports as failing, followed by one of them in the
# rendering the Text-Fabric app provides. The printed line gives the published surface form, the
# form the words spell, and the analysis of each word. The purpose is to make the disagreement
# inspectable. The second table of test 1 already characterises the failures as a population.

for _c, _g, _target, _built in group_failures[:4]:
    print(f"{_c:<16} word group {_g}: published {_target!r}, its words spell {_built!r}")
    print(f"{'':<16}   words: "
          + str([(F.voc.v(w), F.lex.v(w), F.sp.v(w), F.source.v(w)) for w in L.d(_g, otype='word')]))

_c, _g, _target, _built = next((f for f in group_failures if f[0] == "neofiti_full"),
                               group_failures[0])
print(f"\n{_c}: word group {_g}")
A.pretty(_g, withNodes=True)

frag_targ        word group 572705: published '', its words spell 'CPVJmWCVRJmDJJNJnWSRBKNJn'
                   words: [('CPVJm', None, None, 'none')]
frag_targ        word group 573763: published '', its words spell 'WHDS'
                   words: [('WHDS', None, None, 'none')]
frag_targ        word group 579299: published 'MJJTWJn', its words spell 'MJJTWJ'
                   words: [('MJJT', 'MJJTWJn', 'adjv', 'word')]
frag_targ        word group 580192: published '', its words spell 'BJT'
                   words: [('BJT', None, None, 'none')]

neofiti_full: word group 809505


In [7]:
# TEST 2. Composition of a segment from the words it contains.
#
# Method: for every `segment` node, concatenate the `voc` feature of the `word` nodes it contains
# and compare the result to the segment's own `segment` feature under the letters comparison.
#
# It used to concatenate the `g_voc` of the word groups instead, and that is now the wrong tier to
# ask at. `g_voc` is the whole CAL word; a word cut by an editorial boundary contributes only its
# own characters to each side, so a segment holding half a word would be compared against the whole
# of it and fail by construction. The word tier carries exactly the characters of the segment it
# sits in, which is what the rung is about. `whole_groups` keeps the old group-level statement for
# comparison, and it is the one that cannot hold wherever a word straddles two states.
#
# `all_readings` uses every word in the segment; `base_reading` uses only the words of groups whose
# `reading` is 0, the underlying reading the first scribe wrote. A segment holding no variant
# satisfies the two identically.
#
# What it measures: whether the word tier accounts for the editorial segment that contains it. This
# is the rung W2 was written for: attributing a word to the segment its own span falls in, and
# splitting it where the span crosses, took it from 84.16% to the figure below.

test2 = defaultdict(Counter)
for _k, _vns in verse_nodes.items():
    _c = corpus_of_verse[_k]
    for _v in _vns:
        for _s in L.d(_v, otype="segment"):
            gs = L.d(_s, otype="word_group")
            ws = L.d(_s, otype="word")
            target = letters(F.segment.v(_s))
            every = letters("".join(F.voc.v(w) or "" for w in ws))
            base = letters("".join(F.voc.v(w) or "" for g in gs if F.reading.v(g) == 0
                                   for w in L.d(g, otype="word")))
            whole = letters("".join(F.g_voc.v(g) or "" for g in gs))
            _r = test2[_c]
            _r["segments"] += 1
            _r["with_words"] += bool(ws)
            _r["no_letters"] += not target
            _r["has_variants"] += any(F.reading.v(g) != 0 for g in gs)
            _r["filler_only"] += bool(ws) and all(F.source.v(w) == "segment" for w in ws)
            _r["all_readings"] += every == target
            _r["base_reading"] += base == target
            _r["either"] += (every == target or base == target)
            _r["whole_groups"] += whole == target

table("TEST 2  the words of a segment spell the segment", test2,
      ["segments", "with_words", "no_letters", "has_variants", "filler_only",
       "all_readings", "base_reading", "either", "whole_groups"],
      pct=[("% either", "either", "segments"),
           ("% whole groups", "whole_groups", "segments")])
results["2 words compose the segment"] = (sum(test2[c]["segments"] for c in CORPORA),
                                          sum(test2[c]["either"] for c in CORPORA))


TEST 2  the words of a segment spell the segment
            corpus  segments  with_words  no_letters  has_variants  filler_only  all_readings  base_reading  either  whole_groups  % either  % whole groups
  ----------------  --------  ----------  ----------  ------------  -----------  ------------  ------------  ------  ------------  --------  --------------
     cairo_genizah     2,423       2,423          10             0            0         2,313         2,313   2,313           815     95.46           33.64
         frag_targ     2,974       2,974           1             4            0         2,907         2,903   2,907         1,714     97.75           57.63
  neofiti_base_raw     9,635       9,635          11             0            0         9,585         9,585   9,585         5,663     99.48           58.78
      neofiti_full    33,228      33,228          26         7,985            2        33,009        25,092  33,042        29,045     99.44           87.41
               

In [8]:
# TEST 3. Reconstruction of the verse line from the word group tier.
#
# Method: concatenate the `g_voc` feature of every word group of a verse, in publication order,
# and compare the result to the verse's `line` feature under the letters comparison. Variant
# readings are included, because the published line carries them inline.
#
# A word group is taken by `id` and once only. A group an editorial boundary cuts in two publishes
# as two nodes carrying the same `g_voc`, so concatenating over nodes would emit that word's
# surface twice and fail the verse for a reason that is not about the word group tier.
#
# What it measures: whether the surface forms recorded at the word group tier account for the
# verse line the dataset publishes. This is the strongest statement available at that tier, and
# it is independent of the morphological decomposition tested in test 1.

test3 = defaultdict(Counter)
for _k, _vns in verse_nodes.items():
    _r = test3[corpus_of_verse[_k]]
    line = max({F.line.v(v) or "" for v in _vns})
    _seen, _built = set(), []
    for _v in _vns:
        for _g in L.d(_v, otype="word_group"):
            _gid = F.id.v(_g)
            if _gid in _seen:
                continue
            _seen.add(_gid)
            _built.append(F.g_voc.v(_g) or "")
    _r["verses"] += 1
    _r["word_groups"] += len(_seen)
    _r["groups_spell_line"] += letters("".join(_built)) == letters(line)

table("TEST 3  the word groups of a verse spell the verse line", test3,
      ["verses", "word_groups", "groups_spell_line"],
      pct=[("%", "groups_spell_line", "verses")])
results["3 word groups compose the verse line"] = (sum(test3[c]["verses"] for c in CORPORA),
                                                   sum(test3[c]["groups_spell_line"] for c in CORPORA))


TEST 3  the word groups of a verse spell the verse line
            corpus  verses  word_groups  groups_spell_line      %
  ----------------  ------  -----------  -----------------  -----
     cairo_genizah     560       12,451                536  95.71
         frag_targ   1,525       32,012              1,478  96.92
  neofiti_base_raw   5,827       96,108              5,371  92.17
      neofiti_full   5,844      115,679              5,644  96.58
               psj   5,831      104,763              5,809  99.62
  ----------------  ------  -----------  -----------------  -----
             TOTAL  19,587      361,013             18,838  96.18


In [9]:
# TEST 4. Reconstruction of the verse line from the segment tier.
#
# Method: concatenate the `segment` feature of every `segment` node of a verse, in publication
# order, and compare the result to the verse's `line` feature under the letters comparison.
# Variant segments are included, because the published line carries the competing readings
# inline, and restricting the concatenation to the segments that are not variants recovers no
# verse the full concatenation loses. `utf8_spell_line` repeats the comparison in Aramaic square
# script. `repeated` counts the segments a verse publishes more than once under one `id`, whose
# text the concatenation therefore emits twice.
#
# The same loop resolves the segment offsets against the line, which the cell below reports: the
# line cut at a segment's span is compared to the segment's own string, and the spans of a verse
# are then examined as a partition — they must not overlap, the material between two consecutive
# spans must be whitespace alone, and the material before the first and after the last must
# likewise be whitespace. A verse satisfying all three has segments that tile the line. Whether
# the spans also ascend in publication order is test 6's question, where it is asked of every
# tier at once. The word tier carries `start_idx` and `end_idx` of its own, and the cell after
# the next one resolves those: a segment's span is the minimum and the maximum of its words',
# so the two tiers cannot disagree and the word half of the statement is the stronger one.
#
# What it measures: whether the editorial tier accounts for the verse line the dataset publishes.
# Test 3 asks that of the word group tier and test 5 of the word tier. The segment is the tier
# between them, and the only one whose strings are runs of the line rather than morphology.
#
# Interpretation: the second table restricts the population by the morphology underneath. A
# segment carries morphology when every word it holds has `source` `word`; a verse is analysed
# when every one of its segments does. That is the population the decomposition is answerable
# for, and it is reported beside the global figure and beside the rate for the verses that hold
# an unanalysed segment, so the three can be read apart, as tests 1 and 5 do at their own tiers.
# The third table attributes the failures. `text_differs_from_its_span` counts the verses holding
# a segment whose stored string is not the slice of the line its own offsets name, which is the
# containment boundary test 2 measures from the other side. `spans_still_cover_the_line` counts
# the failures whose geometry does return the whole line, so that the loss is in the strings and
# not in the spans. The fourth gives the proportion of the letters of each line that the segments
# emit at all; it is high everywhere, so the shortfall is many small local losses rather than
# missing material.

test4 = defaultdict(Counter)
segment_failures = []
for _k, _vns in verse_nodes.items():
    _r = test4[corpus_of_verse[_k]]
    line = max({F.line.v(v) or "" for v in _vns})
    line_utf8 = max({F.line_utf8.v(v) or "" for v in _vns})
    segs = [s for v in _vns for s in L.d(v, otype="segment")]
    target = letters(line)
    built = letters("".join(F.segment.v(s) or "" for s in segs))
    ok = built == target
    analysed, spans = [], set()
    for s in segs:
        ws = L.d(s, otype="word")
        analysed.append(bool(ws) and all(F.source.v(w) == "word" for w in ws))
        a, b = F.start_idx.v(s), F.end_idx.v(s)
        if a is None:
            _r["segment_no_offset"] += 1
            continue
        _r["segment_offset"] += 1
        _r["segment_slice_is_segment"] += line[a - 1:b] == (F.segment.v(s) or "")
        spans.add((a, b))
    every_analysed = bool(analysed) and all(analysed)
    _r["verses"] += 1
    _r["segments"] += len(segs)
    _r["repeated"] += len(segs) - len({F.id.v(s) for s in segs})
    _r["segments_spell_line"] += ok
    _r["utf8_spell_line"] += letters(
        "".join(F.segment_utf8.v(s) or "" for s in segs)) == letters(line_utf8)
    _r["line_letters"] += len(target)
    _r["segment_letters"] += len(built)
    _r["with_morphology"] += sum(analysed)
    _r["without_morphology"] += len(analysed) - sum(analysed)
    _r["analysed_verses"] += every_analysed
    _r["analysed_spell_line"] += every_analysed and ok
    _r["rest_verses"] += not every_analysed
    _r["rest_spell_line"] += (not every_analysed) and ok
    if spans:
        # A verse whose rows are not contiguous under the publication order is emitted as more
        # than one verse node, and a segment spanning the break is emitted once per node. The
        # partition is a statement about distinct spans, so they are collected as a set.
        ordered = sorted(spans)
        disjoint = all(nxt[0] > cur[1] for cur, nxt in zip(ordered, ordered[1:]))
        gaps_ws = all(not line[cur[1]:nxt[0] - 1].strip() for cur, nxt in zip(ordered, ordered[1:]))
        edges_ws = not line[:ordered[0][0] - 1].strip() and not line[ordered[-1][1]:].strip()
        _r["verses_with_spans"] += 1
        _r["spans_disjoint"] += disjoint
        _r["gaps_are_whitespace"] += gaps_ws
        _r["edges_are_whitespace"] += edges_ws
        _r["partitions_the_line"] += disjoint and gaps_ws and edges_ws
    if not ok:
        segment_failures.append((corpus_of_verse[_k], _k, target, built))
        _r["differ"] += 1
        _r["reordered"] += sorted(built) == sorted(target)
        _r["segments_shorter"] += len(built) < len(target)
        _r["segments_longer"] += len(built) > len(target)
        _r["text_differs_from_its_span"] += any(
            F.start_idx.v(s) is not None
            and line[F.start_idx.v(s) - 1:F.end_idx.v(s)] != (F.segment.v(s) or "")
            for s in segs)
        _r["spans_still_cover_the_line"] += letters(
            "".join(line[a - 1:b] for a, b in sorted(spans))) == target
        _r["segment_without_a_span"] += any(F.start_idx.v(s) is None for s in segs)

table("TEST 4  the segments of a verse spell the verse line", test4,
      ["verses", "segments", "repeated", "segments_spell_line", "utf8_spell_line"],
      pct=[("%", "segments_spell_line", "verses"), ("% utf8", "utf8_spell_line", "verses")])
print()
table("TEST 4  by the morphology under the segment", test4,
      ["segments", "with_morphology", "without_morphology", "analysed_verses",
       "analysed_spell_line", "rest_verses", "rest_spell_line"],
      pct=[("% segments analysed", "with_morphology", "segments"),
           ("% of analysed verses", "analysed_spell_line", "analysed_verses"),
           ("% of the rest", "rest_spell_line", "rest_verses")])
print()
table("TEST 4  the composition of the failures", test4,
      ["differ", "reordered", "segments_shorter", "segments_longer",
       "text_differs_from_its_span", "spans_still_cover_the_line", "segment_without_a_span"])
print()
table("TEST 4  how much of each line the segments emit", test4,
      ["line_letters", "segment_letters"],
      pct=[("% of the letters", "segment_letters", "line_letters")])
results["4 segments compose the verse line"] = (sum(test4[c]["verses"] for c in CORPORA),
                                                sum(test4[c]["segments_spell_line"]
                                                    for c in CORPORA))

TEST 4  the segments of a verse spell the verse line
            corpus  verses  segments  repeated  segments_spell_line  utf8_spell_line      %  % utf8
  ----------------  ------  --------  --------  -------------------  ---------------  -----  ------
     cairo_genizah     560     2,423        36                  540              490  96.43   87.50
         frag_targ   1,525     2,974         8                1,512            1,453  99.15   95.28
  neofiti_base_raw   5,827     9,635        22                5,800            5,631  99.54   96.64
      neofiti_full   5,844    33,228        97                5,785            5,600  98.99   95.82
               psj   5,831     6,937         5                5,820            5,779  99.81   99.11
  ----------------  ------  --------  --------  -------------------  ---------------  -----  ------
             TOTAL  19,587    55,197       168               19,457           18,953  99.34   96.76

TEST 4  by the morphology under the segment
  

In [10]:
# The same segments, read as geometry rather than as strings: the two tables the offsets still
# support. `% with a span` is the population the second percentage is taken over. A segment whose
# slice returns its own text is one all of whose characters belong to words that segment holds,
# which is the containment property test 2 measures from the other side, and a verse that
# partitions is one whose segments tile its line with nothing but whitespace between them.

table("TEST 4  the segment spans, resolved against the line", test4,
      ["segments", "segment_offset", "segment_no_offset", "segment_slice_is_segment"],
      pct=[("% with a span", "segment_offset", "segments"),
           ("% slice is the segment", "segment_slice_is_segment", "segment_offset")])
print()
table("TEST 4  the segment spans as a partition of the line", test4,
      ["verses", "verses_with_spans", "spans_disjoint", "gaps_are_whitespace",
       "edges_are_whitespace", "partitions_the_line"],
      pct=[("% partitioning", "partitions_the_line", "verses_with_spans")])

TEST 4  the segment spans, resolved against the line
            corpus  segments  segment_offset  segment_no_offset  segment_slice_is_segment  % with a span  % slice is the segment
  ----------------  --------  --------------  -----------------  ------------------------  -------------  ----------------------
     cairo_genizah     2,423           2,321                102                     2,284          95.79                   98.41
         frag_targ     2,974           2,941                 33                     2,903          98.89                   98.71
  neofiti_base_raw     9,635           9,622                 13                     9,513          99.87                   98.87
      neofiti_full    33,228          32,942                286                    32,864          99.14                   99.76
               psj     6,937           6,935                  2                     6,932          99.97                   99.96
  ----------------  --------  --------------

In [11]:
# TEST 5. Reconstruction of the verse line from the word (slot) tier.
#
# Method: the slot type is `word`, so the reconstruction is the text API applied to the verse.
# The format `text-trans-full` is defined as `{voc}{trailer}`, which concatenates each word with
# the material that follows it, and the result is compared to the `line` feature of the verse.
# The comparison is made on the whole string after trimming the trailing space that the final
# word contributes, so it is exact at the level of the character and includes every space.
# `letters_only` repeats the comparison under the letters class, and `utf8_spell_line` repeats
# the exact comparison in Aramaic square script using `text-orig-full` against `line_utf8`.
#
# What it measures: whether the published slot sequence of a verse is the published verse line.
# This is the end to end statement of the dataset and the one a reader consuming the corpus
# through T.text will experience directly.
#
# Interpretation: the second table restricts the population to the verses in which every word
# carries morphology, which separates the verses the decomposition is answerable for from the
# verses where the source material supplies no analysis. The column `words_with_morphology`
# gives the same quantity at the level of the word.

test5 = defaultdict(Counter)
line_failures = []
for _k, _vns in verse_nodes.items():
    _r = test5[corpus_of_verse[_k]]
    line = max({F.line.v(v) or "" for v in _vns})
    line_utf8 = max({F.line_utf8.v(v) or "" for v in _vns})
    ws = [w for v in _vns for w in L.d(v, otype="word")]
    built = "".join((F.voc.v(w) or "") + (F.trailer.v(w) or "") for w in ws)
    built_utf8 = "".join((F.voc_utf8.v(w) or "") + (F.trailer.v(w) or "") for w in ws)
    analysed = all(F.source.v(w) == "word" for w in ws)
    ok = built.strip() == line
    _r["verses"] += 1
    _r["words"] += len(ws)
    _r["words_with_morphology"] += sum(1 for w in ws if F.source.v(w) == "word")
    _r["words_spell_line"] += ok
    _r["letters_only"] += letters(built) == letters(line)
    _r["utf8_spell_line"] += built_utf8.strip() == line_utf8
    _r["every_word_analysed"] += analysed
    _r["analysed_and_spelled"] += analysed and ok
    if not ok:
        line_failures.append((corpus_of_verse[_k], _k, line, built.strip()))

table("TEST 5  the words of a verse spell the verse line, character for character", test5,
      ["verses", "words_spell_line", "letters_only", "utf8_spell_line"],
      pct=[("% exact", "words_spell_line", "verses"), ("% letters", "letters_only", "verses")])
print()
table("TEST 5  restricted to the verses in which every word carries morphology", test5,
      ["verses", "words", "words_with_morphology", "every_word_analysed", "analysed_and_spelled"],
      pct=[("% words", "words_with_morphology", "words"),
           ("% of those verses", "analysed_and_spelled", "every_word_analysed")])
results["5 words compose the verse line"] = (sum(test5[c]["verses"] for c in CORPORA),
                                             sum(test5[c]["words_spell_line"] for c in CORPORA))

TEST 5  the words of a verse spell the verse line, character for character
            corpus  verses  words_spell_line  letters_only  utf8_spell_line  % exact  % letters
  ----------------  ------  ----------------  ------------  ---------------  -------  ---------
     cairo_genizah     560               524           525              476    93.57      93.75
         frag_targ   1,525             1,487         1,487            1,398    97.51      97.51
  neofiti_base_raw   5,827             5,817         5,819            5,069    99.83      99.86
      neofiti_full   5,844             5,775         5,829            5,162    98.82      99.74
               psj   5,831             5,825         5,827            5,639    99.90      99.93
  ----------------  ------  ----------------  ------------  ---------------  -------  ---------
             TOTAL  19,587            19,428        19,487           17,744    99.19      99.49

TEST 5  restricted to the verses in which every word carries

In [12]:
# Repeats test 5 through T.text and classifies its failures, then shows the point at which two
# of them first diverge.
#
# The repetition through the text API serves as a check on the hand written concatenation above.
# It covers the verses occupying a single verse node, because T.text takes one node at a time.
#
# The classification asks which tier is responsible for each failure. A verse with nothing
# published to spell it with holds only words whose `source` is `none`, so the manuscript records
# a line that the source material never analysed and never transcribed at the word tier. A verse
# in which some word carries no morphology is short by that word. A verse in which every word is
# analysed and the reconstruction still differs is the residue attributable to the alignment.

_agree = _differ = 0
for _k, _vns in verse_nodes.items():
    if len(_vns) == 1:
        if T.text(_vns[0], fmt="text-trans-full").strip() == (F.line.v(_vns[0]) or ""):
            _agree += 1
        else:
            _differ += 1
print(f"T.text(verse, fmt='text-trans-full') == line:  {_agree:,} agree, {_differ:,} differ\n")

kinds = Counter()
residue = []
for _c, _k, _line, _built in line_failures:
    ws = [w for v in verse_nodes[_k] for w in L.d(v, otype="word")]
    if not _built:
        kinds["nothing published to spell it with"] += 1
    elif any(F.source.v(w) != "word" for w in ws):
        kinds["some word carries no morphology"] += 1
    else:
        kinds["every word analysed, residue of the alignment"] += 1
        residue.append((_c, _k, _line, _built))
for kind, n in kinds.most_common():
    print(f"  {n:>5,}  {kind}")


def first_divergence(a, b):
    i = next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))
    return i, a[max(0, i - 28):i + 28], b[max(0, i - 28):i + 28]


print()
for _c, _k, _line, _built in residue[:2]:
    at, la, lb = first_divergence(_line, _built)
    print(f"{_c} {_k[0]} uid {_k[1]}, first difference at character {at}")
    print(f"  line : ...{la}...")
    print(f"  words: ...{lb}...")

T.text(verse, fmt='text-trans-full') == line:  19,426 agree, 159 differ

     92  every word analysed, residue of the alignment
     67  some word carries no morphology

frag_targ fragment_targum_v uid 1352, first difference at character 63
  line : ...R C<H XD> H> KL MYR>J MJJTWJn...
  words: ...R C<H XD> H> KL MYR>J MJJTWJ...
frag_targ fragment_targum_v uid 1699, first difference at character 5
  line : ...MXZR> TJXZR LJH JT MCKWN...
  words: ...MXZR>BM<LJ TJXZRCJMC> LJHWJDMWk J...


In [13]:
# The word tier's offsets, stated as coverage and not as a rate over the words that have one.
#
# `sql/views_next` writes `word_etcbc_start_idx` only where the line cut at it already returns the
# word: the CASE that produces the column tests that equality and writes NULL when it fails. So
# "the offsets resolve against the line" is true of every word carrying one *by construction*, and
# quoting that figure alone is quoting a tautology — 100% of a population defined as the part that
# passes. The number that carries information is how many words carry an offset at all, and the two
# are printed together here so the second can never be read without the first.
#
# The resolution rate is still worth measuring, because it is not vacuous in the *dataset*: the
# offset travels from the matview through the publication view, the XML and the TF compiler, and a
# fault in any of those would show up here as a slice that no longer returns its word.
#
# `part` is why a word may legitimately carry no offset: a word cut by an editorial boundary
# publishes one node per side, each holding only its own characters, so the population is pieces
# and not CAL words.

offsets = defaultdict(Counter)
for _k, _vns in verse_nodes.items():
    _r = offsets[corpus_of_verse[_k]]
    line = max({F.line.v(v) or "" for v in _vns})
    for _w in (w for v in _vns for w in L.d(v, otype="word")):
        voc = (F.voc.v(_w) or "").strip()
        _r["words"] += 1
        _r["with_surface"] += bool(voc)
        _r["split_pieces"] += (F.part.v(_w) or 1) > 1
        a, b = F.start_idx.v(_w), F.end_idx.v(_w)
        if a is None:
            _r["no_offset"] += 1
            continue
        _r["with_offset"] += 1
        _r["slice_is_word"] += line[a - 1:b] == voc

table("the word tier's offsets, as coverage", offsets,
      ["words", "with_surface", "split_pieces", "with_offset", "no_offset", "slice_is_word"],
      pct=[("% carrying an offset", "with_offset", "words"),
           ("% of those resolving", "slice_is_word", "with_offset")])
results["word offsets carried by the word tier"] = (
    sum(offsets[c]["words"] for c in CORPORA),
    sum(offsets[c]["with_offset"] for c in CORPORA))
results["word offsets that resolve, of those carried"] = (
    sum(offsets[c]["with_offset"] for c in CORPORA),
    sum(offsets[c]["slice_is_word"] for c in CORPORA))


the word tier's offsets, as coverage
            corpus    words  with_surface  split_pieces  with_offset  no_offset  slice_is_word  % carrying an offset  % of those resolving
  ----------------  -------  ------------  ------------  -----------  ---------  -------------  --------------------  --------------------
     cairo_genizah   18,085        17,414           925       16,151      1,934         16,151                 89.31                100.00
         frag_targ   43,584        43,413           817       41,885      1,699         41,885                 96.10                100.00
  neofiti_base_raw  132,969       132,006         2,373      131,928      1,041        131,928                 99.22                100.00
      neofiti_full  157,316       155,440         2,418      154,490      2,826        154,490                 98.20                100.00
               psj  143,262       142,679           739      142,679        583        142,679                 99.59             

In [14]:
# TEST 6. Agreement of the stored ordinals with the publication order.
#
# Method: within each verse, read the segments and the word groups in node order, which is the
# order the dataset publishes, and count the adjacent pairs whose stored ordinal decreases. For
# segments the ordinals are `rank` and `start_idx`. For word groups the ordinal is the pair
# (`rank`, `reading`), because groups competing for one position share a rank and are ordered
# among themselves by reading. The final column tests that the ranks of the word groups of a
# verse form a dense sequence from zero, which is the property that makes rank usable as a word
# index.
#
# What it measures: whether a consumer reading the corpus in node order and a consumer sorting by
# the stored ordinals obtain the same sequence. An inversion would allow a verse to be published
# in an order its own features contradict, and the preceding tests would not detect it, because
# each of them reads in node order throughout.

test6 = defaultdict(Counter)
for _k, _vns in verse_nodes.items():
    _r = test6[corpus_of_verse[_k]]
    _r["verses"] += 1
    segs = [s for v in _vns for s in L.d(v, otype="segment")]
    ranks = [F.rank.v(s) for s in segs]
    starts = [F.start_idx.v(s) for s in segs]
    _r["segments"] += len(segs)
    _r["segment_rank_inversions"] += sum(
        1 for a, b in zip(ranks, ranks[1:]) if a is not None and b is not None and b < a)
    _r["segment_span_inversions"] += sum(
        1 for a, b in zip(starts, starts[1:]) if a is not None and b is not None and b < a)
    gs = [(F.rank.v(g), F.reading.v(g)) for v in _vns for g in L.d(v, otype="word_group")]
    _r["word_groups"] += len(gs)
    _r["group_order_inversions"] += sum(1 for a, b in zip(gs, gs[1:]) if b < a)
    slots = {rank for rank, _ in gs}
    _r["slot_rank_dense"] += sorted(slots) == list(range(len(slots)))

table("TEST 6  stored ordinals against publication order", test6,
      ["verses", "segments", "segment_rank_inversions", "segment_span_inversions",
       "word_groups", "group_order_inversions", "slot_rank_dense"],
      pct=[("% dense", "slot_rank_dense", "verses")])
results["6 ordinals agree with node order"] = (
    sum(test6[c]["verses"] for c in CORPORA), sum(test6[c]["slot_rank_dense"] for c in CORPORA))

TEST 6  stored ordinals against publication order
            corpus  verses  segments  segment_rank_inversions  segment_span_inversions  word_groups  group_order_inversions  slot_rank_dense  % dense
  ----------------  ------  --------  -----------------------  -----------------------  -----------  ----------------------  ---------------  -------
     cairo_genizah     560     2,423                       16                        1       14,480                       0              560   100.00
         frag_targ   1,525     2,974                        8                        8       33,109                       0            1,525   100.00
  neofiti_base_raw   5,827     9,635                       14                       14       98,835                       0            5,827   100.00
      neofiti_full   5,844    33,228                       57                       52      118,448                       0            5,844   100.00
               psj   5,831     6,937              

In [15]:
# Collects the six tests into one table. The population column states what was counted, which
# differs by test: groups for test 1, segments for test 2, and verses for tests 3 to 6. The table is the summary to consult after a regeneration of the dataset, because a
# change in any row is a change in the data or in the boundary of what the dataset publishes.

print(f"  {'test':<40} {'population':>12} {'pass':>12} {'%':>8}")
print("  " + "-" * 76)
for name, (n, ok) in results.items():
    print(f"  {name:<40} {n:>12,} {ok:>12,} {100.0 * ok / n:>8.2f}")

  test                                       population         pass        %
  ----------------------------------------------------------------------------
  1 words compose the word group                361,013      360,290    99.80
  2 words compose the segment                    55,197       54,775    99.24
  3 word groups compose the verse line           19,587       18,838    96.18
  4 segments compose the verse line              19,587       19,457    99.34
  5 words compose the verse line                 19,587       19,428    99.19
  word offsets carried by the word tier         495,216      487,133    98.37
  word offsets that resolve, of those carried      487,133      487,133   100.00
  6 ordinals agree with node order               19,587       19,587   100.00
